In [0]:
# =============================================================================
# NB12 — Compact physics-based material descriptors vs experimental-only
# =============================================================================
# Context (NB11, XGBoost, 5×5 repeated GroupKFold by DOI, paired):
#   L3_full = 0.069 [−0.28, 0.36];  EXP_ONLY = 0.068 (ΔR² = −0.001 ± 0.24);
#   L1_material_only = −0.403 (CI entirely < 0).
#   → With the 114 elemental-average descriptors, composition adds no
#     detectable transferable signal once experimental conditions are known.
#
# Hypothesis tested here:
#   The 114-feature block is too high-dimensional for 125 papers and mostly
#   acts as a paper fingerprint (composition ≈ paper identity), so it overfits.
#   A COMPACT set of physically motivated descriptors (band gap, band-edge
#   positions, electronegativity, structural factors) may carry transferable
#   signal that the large block hides.
#
# Configurations (same repeated splits as NB11 → directly comparable):
#   EXP_ONLY                 23 experimental features (reference)
#   EXP_plus_COMPACT         EXP + compact material set
#   EXP_plus_COMPACT_noEg    same without measured band gap and Eg-derived edges
#   COMPACT_ONLY             compact material set alone
#   L3_full                  EXP + all 114 material + thermo (NB11 reference)
#
# Also prints a within-paper design diagnostic: how many papers compare
# several distinct compositions (needed for any within-paper analysis).
#
# Inputs:  master, X_material, X_thermo, X_exp_v2
# Outputs: ml_perovskites.compact_nb12, /tmp/nb12_compact.png
# Rules:   encoding/imputation inside fold; DOI never in train and val; SEED=42
# =============================================================================

import warnings
warnings.filterwarnings("ignore")

import subprocess, sys
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.impute  import SimpleImputer
from sklearn.metrics import r2_score
from pyspark.sql import SparkSession

SEED, SCHEMA, N_FOLDS, N_REPEATS, TE_M = 42, "ml_perovskites", 5, 5, 10

spark = SparkSession.builder.getOrCreate()
spark.conf.set("spark.sql.shuffle.partitions", "8")

try:
    import xgboost as xgb
except ImportError:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "xgboost"])
    import xgboost as xgb

# ---------------------------------------------------------------------------
# 1. Load and align (identical to NB11)
# ---------------------------------------------------------------------------
master    = spark.table(f"{SCHEMA}.master").toPandas()
X_mat_raw = spark.table(f"{SCHEMA}.X_material").toPandas()
X_thm_raw = spark.table(f"{SCHEMA}.X_thermo").toPandas()
X_exp_raw = spark.table(f"{SCHEMA}.X_exp_v2").toPandas()

def is_use_h2(s):
    return s.astype(str).str.strip().str.lower().isin(["true", "1"])

master["y_log10_1p_final"] = pd.to_numeric(master["y_log10_1p_final"], errors="coerce")
master["bandgap_eV_meas"]  = pd.to_numeric(master["bandgap_eV"], errors="coerce")
df_base = master[is_use_h2(master["use_h2"]) &
                 master["y_log10_1p_final"].notna() &
                 master["doi_key"].notna()].copy()
df_base["y"] = df_base["y_log10_1p_final"].astype(float)

for t in [df_base, X_mat_raw, X_thm_raw, X_exp_raw]:
    t["source_row_id"] = t["source_row_id"].astype(str).str.strip()

META = {"source_row_id", "base_compound", "family_proposed", "formula_base_paper",
        "A_site_paper", "B_site_paper", "doi", "doi_key", "y_log10_1p_final",
        "y_log10_1p", "h2_rate_umol_g_h", "h2_umol_h_rate_mass"}
for raw in [X_mat_raw, X_thm_raw]:
    for c in raw.columns:
        if c != "source_row_id":
            raw[c] = pd.to_numeric(raw[c], errors="coerce")

CAT_COLS = [c for c in ["co_catalyst_type", "run_type", "scavenger_class",
                        "lamp_type", "prep_class"] if c in X_exp_raw.columns]
x_mat = [c for c in X_mat_raw.columns if c not in META
         and pd.api.types.is_numeric_dtype(X_mat_raw[c])
         and X_mat_raw[c].isna().mean() < 0.50]
x_thm = [c for c in X_thm_raw.columns if c not in META
         and pd.api.types.is_numeric_dtype(X_thm_raw[c])
         and X_thm_raw[c].isna().mean() < 0.99]
x_exp_num = [c for c in X_exp_raw.columns if c not in ["source_row_id"] + CAT_COLS]
for c in x_exp_num:
    X_exp_raw[c] = pd.to_numeric(X_exp_raw[c], errors="coerce")
x_exp = x_exp_num + CAT_COLS

df = (df_base[["source_row_id", "doi_key", "base_compound", "y", "bandgap_eV_meas"]]
      .merge(X_mat_raw[["source_row_id"] + x_mat], on="source_row_id", how="left")
      .merge(X_thm_raw[["source_row_id"] + x_thm], on="source_row_id", how="left")
      .merge(X_exp_raw[["source_row_id"] + x_exp], on="source_row_id", how="left"))

# ---------------------------------------------------------------------------
# 2. Compact physics-based material set (only columns that exist are used)
# ---------------------------------------------------------------------------
EG_BASED = ["bandgap_eV_meas", "ECB_butler", "EVB_butler"]
COMPACT_WISHLIST = EG_BASED + [
    "chi_mulliken", "goldschmidt_t", "octahedral_mu", "Ra_Rb_ratio",
    "sigma_r_A", "sigma_r_B", "wav_gs_bandgap", "wav_en_pauling",
    "wav_ionic_radius_shannon_vi", "wav_num_d_valence", "wav_mendeleev_number",
    "wav_first_ion_en", "wav_electron_affinity",
]
COMPACT = [c for c in COMPACT_WISHLIST if c in df.columns]
missing = [c for c in COMPACT_WISHLIST if c not in df.columns]
print(f"Compact set ({len(COMPACT)}): {COMPACT}")
if missing:
    print(f"  not found (skipped): {missing}")
print("Fill rates:", {c: f"{df[c].notna().mean()*100:.0f}%" for c in COMPACT})

CONFIGS = {
    "EXP_ONLY":              x_exp,
    "EXP_plus_COMPACT":      x_exp + COMPACT,
    "EXP_plus_COMPACT_noEg": x_exp + [c for c in COMPACT if c not in EG_BASED],
    "COMPACT_ONLY":          COMPACT,
    "L3_full":               x_mat + x_thm + x_exp,
}

# ---------------------------------------------------------------------------
# 3. Within-paper design diagnostic (descriptive only, no modelling)
# ---------------------------------------------------------------------------
per_doi = df.groupby("doi_key").agg(n_rows=("y", "size"),
                                    n_compounds=("base_compound", "nunique"),
                                    y_sd=("y", "std"))
multi = per_doi[per_doi["n_compounds"] >= 2]
print("\n--- Within-paper design diagnostic ---")
print(f"  Papers: {len(per_doi)} | with ≥2 distinct compounds: {len(multi)} "
      f"({len(multi)/len(per_doi)*100:.0f}%) covering {multi['n_rows'].sum()} rows")
print(f"  Distinct compounds per paper: median {per_doi['n_compounds'].median():.0f}, "
      f"max {per_doi['n_compounds'].max()}")
print(f"  Within-paper SD of y (papers with ≥2 compounds): "
      f"median {multi['y_sd'].median():.2f} log units")

# ---------------------------------------------------------------------------
# 4. Repeated grouped CV (same seeds → same splits as NB11)
# ---------------------------------------------------------------------------
dois = np.array(sorted(df["doi_key"].unique()))

def grouped_splits(repeat):
    perm = np.random.default_rng(SEED + repeat).permutation(dois)
    fold_of = {d: i % N_FOLDS for i, d in enumerate(perm)}
    f = df["doi_key"].map(fold_of).values
    return [(np.where(f != k)[0], np.where(f == k)[0]) for k in range(N_FOLDS)]

def encode(tr, va, cols):
    Xtr, Xva = tr[cols].copy(), va[cols].copy()
    for cc in [c for c in CAT_COLS if c in cols]:
        gm = tr["y"].mean()
        st = tr.groupby(cc)["y"].agg(["mean", "count"])
        enc = ((st["mean"] * st["count"] + gm * TE_M) / (st["count"] + TE_M)).to_dict()
        Xtr[cc] = tr[cc].map(enc).fillna(gm)
        Xva[cc] = va[cc].map(enc).fillna(gm)
    return Xtr.astype(float), Xva.astype(float)

def model():
    return xgb.XGBRegressor(n_estimators=400, learning_rate=0.05, max_depth=4,
                            subsample=0.8, colsample_bytree=0.7, min_child_weight=3,
                            reg_lambda=1.0, tree_method="hist",
                            random_state=SEED, verbosity=0)

rows = []
for rep in range(N_REPEATS):
    for k, (tr_i, va_i) in enumerate(grouped_splits(rep)):
        tr, va = df.iloc[tr_i], df.iloc[va_i]
        assert not set(tr["doi_key"]) & set(va["doi_key"]), "DOI leakage!"
        for name, cols in CONFIGS.items():
            Xtr, Xva = encode(tr, va, cols)
            imp = SimpleImputer(strategy="median")
            m = model()
            m.fit(imp.fit_transform(Xtr.values), tr["y"].values)
            yp = m.predict(imp.transform(Xva.values))
            rows.append({"repeat": rep, "fold": k, "config": name,
                         "n_features": len(cols), "r2_log": r2_score(va["y"], yp)})
    print(f"  repeat {rep + 1}/{N_REPEATS} done")

res = pd.DataFrame(rows)
ps  = res.pivot_table(index=["repeat", "fold"], columns="config", values="r2_log")

summary = []
for c in ps.columns:
    d = (ps[c] - ps["EXP_ONLY"]).values
    summary.append({"config": c, "r2_mean": ps[c].mean(),
                    "r2_p2.5": np.percentile(ps[c], 2.5),
                    "r2_p97.5": np.percentile(ps[c], 97.5),
                    "delta_vs_EXP": d.mean(),
                    "delta_p2.5": np.percentile(d, 2.5),
                    "delta_p97.5": np.percentile(d, 97.5),
                    "frac_splits_better_than_EXP": (d > 0).mean()})
summary = pd.DataFrame(summary).sort_values("r2_mean", ascending=False)

pd.set_option("display.width", 200)
print(f"\n--- XGBoost, {N_REPEATS}×{N_FOLDS} grouped CV by DOI — paired vs EXP_ONLY ---")
print(summary.round(3).to_string(index=False))
print("\nReading guide:")
print("  * EXP_plus_COMPACT delta > 0 with CI above 0 → compact material physics transfers.")
print("  * noEg ≈ EXP_ONLY but COMPACT > EXP_ONLY     → the signal is the measured band gap.")
print("  * COMPACT_ONLY vs L1 (NB11: −0.40)           → was the large block just overfitting?")

# ---------------------------------------------------------------------------
# 5. Plot + write
# ---------------------------------------------------------------------------
s = summary.set_index("config")
order = list(s.sort_values("r2_mean").index)
fig, ax = plt.subplots(figsize=(8, 0.55 * len(order) + 1.5))
ax.errorbar(s.loc[order, "r2_mean"], range(len(order)),
            xerr=[s.loc[order, "r2_mean"] - s.loc[order, "r2_p2.5"],
                  s.loc[order, "r2_p97.5"] - s.loc[order, "r2_mean"]],
            fmt="o", capsize=3)
ax.axvline(0, color="black", lw=0.8)
ax.set_yticks(range(len(order))); ax.set_yticklabels(order)
ax.set_xlabel("R² (log) — XGBoost, repeated GroupKFold by DOI (2.5–97.5%)")
plt.tight_layout()
plt.savefig("/tmp/nb12_compact.png", dpi=120)
plt.close()
print("Saved: /tmp/nb12_compact.png")

spark.createDataFrame(res).write.format("delta").mode("overwrite") \
    .option("overwriteSchema", "true").saveAsTable(f"{SCHEMA}.compact_nb12")
print(f"✓ Written: {SCHEMA}.compact_nb12 ({len(res)} rows)")
print("\n=== NB12 COMPLETE ===")